In [ ]:
import pandas as pd
import numpy as np
import os

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.lines import Line2D

In [ ]:
result_folder = "result path"

# all outputs (plots, TB1 csvs) are written under output_dir -- change this one
# path to redirect every save in the notebook
output_dir = "{setup your output path here}/manuscript"
plots_dir = os.path.join(output_dir, "plots")
tb1_dir = os.path.join(output_dir, "tb1")
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(tb1_dir, exist_ok=True)

In [ ]:
# fixed canonical order -- never edited when excluding a site, so numbering
# doesn't reshuffle when EXCLUDED_SITES changes
SITE_ORDER = ['hopkins', 'nu', 'ohsu', 'penn', 'rush', 'ucmc', 'umn', 'michigan', 'ucsf']

# sites to drop from every result below; remaining sites renumber 1..N with no
# gaps. Remove a site from this list to bring it back.
EXCLUDED_SITES = ['penn']

site_name_mapping = {
    site: f'site {i + 1}'
    for i, site in enumerate(s for s in SITE_ORDER if s not in EXCLUDED_SITES)
}

def map_site_name(site):
    # case-insensitive lookup -- source files don't agree on site-name casing
    return site_name_mapping.get(str(site).lower())

def exclude_sites(df, site_col='_site_name'):
    return df[~df[site_col].astype(str).str.lower().isin(EXCLUDED_SITES)].copy()

In [ ]:
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 11

# shared forest-plot panel: draws one point+CI row per metric per site, used by
# sections 1, 2, and 3 below. df must have a 'site_label' column and metrics is
# a list of (metric, ci_low_col, ci_high_col, color).
def plot_ci_panel(ax, df, metrics, group_spacing=1.4, dot_gap=0.4, label_fontsize=14):
    y = np.arange(len(df)) * group_spacing
    for i, (metric, lo_col, hi_col, color) in enumerate(metrics):
        values = df[metric]
        lower_err = (values - df[lo_col]).clip(lower=0)
        upper_err = (df[hi_col] - values).clip(lower=0)
        offset = (i - (len(metrics) - 1) / 2) * dot_gap
        ax.errorbar(
            values, y + offset, xerr=[lower_err, upper_err],
            fmt='o', color=color, ecolor=color,
            elinewidth=1.5, capthick=1.5, capsize=4, markersize=6,
        )
        for yi, v, ue, lo, hi in zip(y + offset, values, upper_err, df[lo_col], df[hi_col]):
            label = f'{v * 100:.0f} [{lo * 100:.0f}-{hi * 100:.0f}]'
            ax.text(v + ue + 0.02, yi, label, ha='left', va='center', fontsize=label_fontsize)

    ax.set_yticks(y)
    ax.set_yticklabels(df['site_label'], fontsize=label_fontsize)
    ax.invert_yaxis()
    ax.set_ylabel('Health System', fontsize=label_fontsize)
    ax.tick_params(axis='x', labelsize=label_fontsize)
    ax.set_xticks(np.arange(0, 1.01, 0.2))
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x * 100:.0f}'))
    ax.set_xlim(0, 1.35)
    ax.grid(False)
    for spine in ('top', 'right'):
        ax.spines[spine].set_visible(False)

SENSITIVITY_COLOR = '#2a78d6'
SPECIFICITY_COLOR = '#b23b3b'

# 1. Prone ICD validation

In [ ]:
proned_cf = exclude_sites(pd.read_csv(os.path.join(result_folder, "proned_cf_all_sites.csv")))

In [ ]:
# pivot site-reported metrics (one row per site) and flatten the CI columns
site_metrics_ci = proned_cf.pivot(index='_site_name', columns='metric', values=['estimate', 'ci_lower', 'ci_upper'])
site_metrics_ci.columns = [
    f'{metric}_ci_low' if stat == 'ci_lower' else f'{metric}_ci_high' if stat == 'ci_upper' else metric
    for stat, metric in site_metrics_ci.columns
]
site_metrics_ci

In [ ]:
# plot sensitivity and specificity with 95% CI, one row per site
plot_df = site_metrics_ci.copy()
plot_df['site_label'] = plot_df.index.map(map_site_name).str.capitalize()
plot_df = plot_df.sort_values(
    'site_label', key=lambda s: s.str.extract(r'(\d+)')[0].astype(int)
)

metrics = [
    ('sensitivity', 'sensitivity_ci_low', 'sensitivity_ci_high', SENSITIVITY_COLOR),
    ('specificity', 'specificity_ci_low', 'specificity_ci_high', SPECIFICITY_COLOR),
]

fig, ax = plt.subplots(figsize=(9, 6.3), dpi=200)
plot_ci_panel(ax, plot_df, metrics, label_fontsize=16)

legend_handles = [
    Line2D([0], [0], marker='o', linestyle='-', color=SENSITIVITY_COLOR, label='Sensitivity (95% CI)'),
    Line2D([0], [0], marker='o', linestyle='-', color=SPECIFICITY_COLOR, label='Specificity (95% CI)'),
]
ax.legend(
    handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, -0.12),
    ncol=2, frameon=False, fontsize=14
)

fig.tight_layout()
fig.savefig(os.path.join(plots_dir, "proned_sensitivity_specificity_by_site.pdf"), bbox_inches="tight")
plt.show()

# 2. Ventilation ICD validation

In [ ]:
vent_cf = exclude_sites(pd.read_csv(os.path.join(result_folder, "vent_cf_all_sites.csv")))

In [ ]:
# pivot site-reported metrics (one row per site) and flatten the CI columns
# (specificity is all NA for ventilation -- excluded from the plot below)
vent_site_metrics_ci = vent_cf.pivot(index='_site_name', columns='metric', values=['estimate', 'ci_lower', 'ci_upper'])
vent_site_metrics_ci.columns = [
    f'{metric}_ci_low' if stat == 'ci_lower' else f'{metric}_ci_high' if stat == 'ci_upper' else metric
    for stat, metric in vent_site_metrics_ci.columns
]
vent_site_metrics_ci

In [ ]:
# plot sensitivity with 95% CI, one row per site
vent_plot_df = vent_site_metrics_ci.copy()
vent_plot_df['site_label'] = vent_plot_df.index.map(map_site_name).str.capitalize()
vent_plot_df = vent_plot_df.sort_values(
    'site_label', key=lambda s: s.str.extract(r'(\d+)')[0].astype(int)
)

vent_metrics = [('sensitivity', 'sensitivity_ci_low', 'sensitivity_ci_high', SENSITIVITY_COLOR)]

fig, ax = plt.subplots(figsize=(9, 6.3), dpi=200)
plot_ci_panel(ax, vent_plot_df, vent_metrics, label_fontsize=16)

legend_handles = [
    Line2D([0], [0], marker='o', linestyle='-', color=SENSITIVITY_COLOR, label='Sensitivity (95% CI)'),
]
ax.legend(
    handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, -0.12),
    ncol=1, frameon=False, fontsize=14
)

fig.tight_layout()
fig.savefig(os.path.join(plots_dir, "ventilation_sensitivity_specificity_by_site.pdf"), bbox_inches="tight")
plt.show()

# 3. Combined sensitivity/specificity figure

In [ ]:
# prone ICD (A) and ventilation ICD (B) side by side
fig, (ax_prone, ax_vent) = plt.subplots(1, 2, figsize=(16, 6.3), dpi=200)

panels = [
    (ax_prone, plot_df, metrics, 'A. Prone positioning ICD codes'),
    (ax_vent, vent_plot_df, vent_metrics, 'B. Invasive mechanical ventilation ICD codes'),
]

for ax, df, panel_metrics, title in panels:
    plot_ci_panel(ax, df, panel_metrics)
    ax.text(-0.14, 1.12, title, transform=ax.transAxes, fontsize=16, fontweight='bold', ha='left', va='bottom')

legend_handles = [
    Line2D([0], [0], marker='o', linestyle='-', color=SENSITIVITY_COLOR, label='Sensitivity (95% CI)'),
    Line2D([0], [0], marker='o', linestyle='-', color=SPECIFICITY_COLOR, label='Specificity (95% CI)'),
]
fig.legend(
    handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, 0.02),
    ncol=2, frameon=False, fontsize=14
)

fig.tight_layout()
fig.savefig(os.path.join(plots_dir, "sensitivity_specificity_combined.pdf"), bbox_inches="tight")
plt.show()

# 4. Table 1 -- pooled across sites

In [ ]:
tb1_cat = exclude_sites(pd.read_csv(os.path.join(result_folder, "tableone_cat_all_sites.csv")))
tb1_cont = exclude_sites(pd.read_csv(os.path.join(result_folder, "tableone_cont_all_sites.csv")))

# sites mask freq as the literal string "<10" when 0 < freq < 10 (small-cell
# suppression), leaving percent/cum.percent blank. Replace with the numeric
# midpoint (5) so freq stays usable downstream (e.g. the pooled-count sums);
# percent is recomputed from freq/n rather than trusted from the source file.
SMALL_CELL_MASK = r'(?i)^\s*n?\s*<\s*10\s*$'

def unmask_small_cell(df, col, replacement=5, label=""):
    mask = df[col].astype(str).str.match(SMALL_CELL_MASK)
    if mask.any():
        print(f"INFO: {label}['{col}'] unmasked {mask.sum()} small-cell ('<10') value(s) -> {replacement}")
        df.loc[mask, col] = replacement
    return df

tb1_cat = unmask_small_cell(tb1_cat, "freq", label="tb1_cat")

# site-submitted stat columns can carry non-numeric placeholders (e.g. "<11",
# "NA") that leave the column as object dtype -- coerce to numeric and warn
# about anything that couldn't be parsed instead of silently becoming 0.
def coerce_numeric(df, cols, label):
    for col in cols:
        raw = df[col]
        numeric = pd.to_numeric(raw, errors="coerce")
        bad = raw[numeric.isna() & raw.notna()]
        if not bad.empty:
            print(f"WARNING: {label}['{col}'] has {len(bad)} non-numeric value(s), coerced to NaN: {sorted(bad.unique())}")
        df[col] = numeric
    return df

tb1_cat = coerce_numeric(tb1_cat, ["n", "miss", "p.miss", "freq", "percent", "cum.percent"], "tb1_cat")
tb1_cont = coerce_numeric(tb1_cont, ["n", "miss", "p.miss", "mean", "sd", "median", "p25", "p75", "min", "max", "skew", "kurt"], "tb1_cont")

In [ ]:
# TB1 variable config -- shared by the pooled (Section 4) and per-site (Section 5) builds
strata_order = ['Overall', 'not_proned', 'proned']
target_vars = ['age_at_admission', 'bmi']
range_vars = [
    'min_pf_ratio', 'first_proseva_peep', 'first_proseva_fio2',
    'sofa_score', 'median_pt_prone_duration', 'vent_duration_hours',
]
range_decimals = {
    'min_pf_ratio': 1,
    'first_proseva_peep': 1,
    'first_proseva_fio2': 2,
    'sofa_score': 0,
    'median_pt_prone_duration': 1,
    'vent_duration_hours': 1,
}
cat_vars = ['sex_category', 'race_category', 'ethnicity_category', 'death_or_hospice', 'eligible_by_proseva', 'eligible_by_prone']
binary_vars = ['death_or_hospice', 'eligible_by_proseva', 'eligible_by_prone']
# multi-level vars get one row per level; binary_vars get a single "1" row
multi_level_vars = ['sex_category', 'race_category', 'ethnicity_category']
multi_level_labels = {
    'sex_category': 'Gender',
    'race_category': 'Race',
    'ethnicity_category': 'Ethnicity',
}

# Section 4 row labels -- range_vars reported as the cross-site range of each site's median
row_labels = {
    'age_at_admission': 'Age, Mean (SD)',
    'bmi': 'BMI, Mean (SD)',
    'min_pf_ratio': 'Min PF Ratio, Median Range',
    'first_proseva_peep': 'First Proseva PEEP, Median Range',
    'first_proseva_fio2': 'First Proseva FiO2, Median Range',
    'sofa_score': 'SOFA Score, Median Range',
    'median_pt_prone_duration': 'Median Prone Duration, Median Range',
    'vent_duration_hours': 'Ventilation Duration Hours, Median Range',
    'death_or_hospice': 'Death or hospice, n (%)',
    'eligible_by_proseva': 'Eligible by PROSEVA, n (%)',
    'eligible_by_prone': 'Eligible by Prone, n (%)',
}

In [ ]:
# pool each site's mean/SD into one cross-site mean/SD, using each site's exact
# eff_n = n - miss, mean, and sd (reconstructs the pooled mean/variance exactly,
# as if from the raw values):
#   pooled_var = [Sigma (eff_n_i - 1) sd_i^2 + Sigma eff_n_i (mean_i - pooled_mean)^2] / (Sigma eff_n_i - 1)
def pool_mean_sd(group):
    eff_n = group['n'] - group['miss']
    pooled_n = eff_n.sum()
    pooled_mean = (group['mean'] * eff_n).sum() / pooled_n
    within_ss = ((eff_n - 1) * group['sd'] ** 2).sum()
    between_ss = (eff_n * (group['mean'] - pooled_mean) ** 2).sum()
    pooled_sd = np.sqrt((within_ss + between_ss) / (pooled_n - 1))
    return pd.Series({'n': pooled_n, 'mean': pooled_mean, 'sd': pooled_sd})

In [ ]:
def build_tb1(cont_df, cat_df):
    # age_at_admission has 0 missing at every site, so its per-site "n" is the
    # true stratum size -- sum across sites for the pooled total patient count.
    cont_f = cont_df[cont_df['strata'].isin(strata_order)].copy()
    total_patients = (
        cont_f[cont_f['variable'] == 'age_at_admission']
        .groupby('strata')['n'].sum()
        .reindex(strata_order)
    )

    pooled_cont = (
        cont_f[cont_f['variable'].isin(target_vars)]
        .groupby(['variable', 'strata'])
        .apply(pool_mean_sd)
        .reset_index()
    )

    table = pd.DataFrame(
        index=['Total patients'] + [row_labels[v] for v in target_vars] + [row_labels[v] for v in range_vars],
        columns=strata_order,
    )
    table.loc['Total patients'] = total_patients.astype(int)
    for var in target_vars:
        for s in strata_order:
            row = pooled_cont[(pooled_cont['variable'] == var) & (pooled_cont['strata'] == s)].iloc[0]
            table.loc[row_labels[var], s] = f"{row['mean']:.1f} ({row['sd']:.1f})"

    # range_vars: min-max of each site's own median. A stratum missing at every
    # site (e.g. not_proned for prone-duration vars) is all-NaN -> min/max is NaN.
    pooled_range = (
        cont_f[cont_f['variable'].isin(range_vars)]
        .groupby(['variable', 'strata'])['median']
        .agg(min='min', max='max')
        .reset_index()
    )
    for var in range_vars:
        dec = range_decimals[var]
        for s in strata_order:
            row = pooled_range[(pooled_range['variable'] == var) & (pooled_range['strata'] == s)].iloc[0]
            table.loc[row_labels[var], s] = 'N/A' if pd.isna(row['min']) else f"{row['min']:.{dec}f}-{row['max']:.{dec}f}"

    # categorical: fix a data-entry typo (penn's race_category has a stray '='
    # in "White, non-H=ispanic"), normalize level casing/underscores so the same
    # category reported differently across sites pools into one level, and
    # merge 'other'/'unknown' into a single group before pooling
    cat_f = cat_df[cat_df['strata'].isin(strata_order) & cat_df['variable'].isin(cat_vars)].copy()
    cat_f['level'] = cat_f['level'].str.replace('=', '', regex=False).str.lower().str.replace('_', ' ', regex=False)
    cat_f['level'] = cat_f['level'].replace({'other': 'other or unknown', 'unknown': 'other or unknown'})
    cat_f = cat_f.groupby(['strata', 'variable', 'level', '_site_name'], as_index=False) \
        .agg(n=('n', 'first'), freq=('freq', 'sum'))

    # flag any variable not reported by every site, so a pooled row is never
    # mistaken for a full estimate
    n_sites_total = cat_df['_site_name'].nunique()
    for var in cat_vars:
        reporting_sites = sorted(cat_f.loc[cat_f['variable'] == var, '_site_name'].unique())
        if len(reporting_sites) < n_sites_total:
            print(f"WARNING: '{var}' is only reported by {len(reporting_sites)}/{n_sites_total} sites: {reporting_sites}")

    # numerator = pooled freq; denominator = Sigma n across sites, deduplicated
    # to one n per site (n repeats on every level row). Missing values are
    # ignored on purpose -- the denominator is total patients, not non-missing.
    pooled_freq = cat_f.groupby(['variable', 'level', 'strata'])['freq'].sum().reset_index()
    total_n = (
        cat_f.drop_duplicates(['variable', 'strata', '_site_name'])
        .groupby(['variable', 'strata'])['n'].sum().reset_index()
    )
    pooled_cat = pooled_freq.merge(total_n, on=['variable', 'strata'])
    pooled_cat['percent'] = pooled_cat['freq'] / pooled_cat['n'] * 100

    def cat_cell(var, level, s):
        match = pooled_cat[(pooled_cat['variable'] == var) & (pooled_cat['level'] == level) & (pooled_cat['strata'] == s)]
        if match.empty:
            return 'N/A'
        row = match.iloc[0]
        return f"{row['freq']:.0f} ({row['percent']:.1f}%)"

    for var in binary_vars:
        table.loc[row_labels[var]] = [cat_cell(var, '1', s) for s in strata_order]
    for var in multi_level_vars:
        for level in sorted(pooled_cat.loc[pooled_cat['variable'] == var, 'level'].unique()):
            table.loc[f'{multi_level_labels[var]}: {level}, n (%)'] = [cat_cell(var, level, s) for s in strata_order]

    return table

In [ ]:
summary_table = build_tb1(tb1_cont, tb1_cat)
summary_table.to_csv(os.path.join(tb1_dir, "tb1_summary.csv"))
summary_table

# 5. Table 1 -- per site

In [ ]:
SITE_COL = "_site_name"

# Section 5 row labels -- range_vars reported as that site's own median [Q1, Q3],
# unlike Section 4's cross-site median range, so kept separate from row_labels
site_row_labels = {
    'age_at_admission': 'Age, Mean (SD)',
    'bmi': 'BMI, Mean (SD)',
    'min_pf_ratio': 'Min PF Ratio, Median [Q1,Q3]',
    'first_proseva_peep': 'First Proseva PEEP, Median [Q1,Q3]',
    'first_proseva_fio2': 'First Proseva FiO2, Median [Q1,Q3]',
    'sofa_score': 'SOFA Score, Median [Q1,Q3]',
    'median_pt_prone_duration': 'Median Prone Duration, Median [Q1,Q3]',
    'vent_duration_hours': 'Ventilation Duration Hours, Median [Q1,Q3]',
    'death_or_hospice': 'Death or hospice, n (%)',
    'eligible_by_proseva': 'Eligible by PROSEVA, n (%)',
    'eligible_by_prone': 'Eligible by Prone, n (%)',
}

In [ ]:
# a single site's row already carries its own mean/sd/median/p25/p75, so no
# cross-site pooling is needed -- read the values straight off the row. The
# same typo fix / level normalization / other-unknown merge from build_tb1
# applies here too, since any site's categorical data can carry the same issues.
def build_site_tb1(cont_df, cat_df):
    cont_f = cont_df[cont_df['strata'].isin(strata_order)].copy()

    def cont_row(var, s):
        match = cont_f[(cont_f['variable'] == var) & (cont_f['strata'] == s)]
        return match.iloc[0] if not match.empty else None

    table = pd.DataFrame(
        index=['Total patients'] + [site_row_labels[v] for v in target_vars] + [site_row_labels[v] for v in range_vars],
        columns=strata_order,
    )

    for s in strata_order:
        row = cont_row('age_at_admission', s)
        table.loc['Total patients', s] = 'N/A' if row is None else int(row['n'])

    # age/bmi: mean (SD)
    for var in target_vars:
        for s in strata_order:
            row = cont_row(var, s)
            table.loc[site_row_labels[var], s] = 'N/A' if row is None else f"{row['mean']:.1f} ({row['sd']:.1f})"

    # everything else: median [Q1, Q3]
    for var in range_vars:
        dec = range_decimals[var]
        for s in strata_order:
            row = cont_row(var, s)
            table.loc[site_row_labels[var], s] = (
                'N/A' if row is None or pd.isna(row['median'])
                else f"{row['median']:.{dec}f} [{row['p25']:.{dec}f}, {row['p75']:.{dec}f}]"
            )

    cat_f = cat_df[cat_df['strata'].isin(strata_order) & cat_df['variable'].isin(cat_vars)].copy()
    cat_f['level'] = cat_f['level'].str.replace('=', '', regex=False).str.lower().str.replace('_', ' ', regex=False)
    cat_f['level'] = cat_f['level'].replace({'other': 'other or unknown', 'unknown': 'other or unknown'})
    cat_f = cat_f.groupby(['strata', 'variable', 'level'], as_index=False).agg(
        n=('n', 'first'), freq=('freq', 'sum')
    )
    cat_f['percent'] = cat_f['freq'] / cat_f['n'] * 100

    def cat_cell(var, level, s):
        match = cat_f[(cat_f['variable'] == var) & (cat_f['level'] == level) & (cat_f['strata'] == s)]
        if match.empty:
            return 'N/A'
        row = match.iloc[0]
        return f"{row['freq']:.0f} ({row['percent']:.1f}%)"

    for var in binary_vars:
        table.loc[site_row_labels[var]] = [cat_cell(var, '1', s) for s in strata_order]

    for var in multi_level_vars:
        for level in sorted(cat_f.loc[cat_f['variable'] == var, 'level'].unique()):
            table.loc[f'{multi_level_labels[var]}: {level}, n (%)'] = [cat_cell(var, level, s) for s in strata_order]

    return table

In [ ]:
all_sites = sorted(set(tb1_cont[SITE_COL].unique()) | set(tb1_cat[SITE_COL].unique()))
n_written = 0

for site in all_sites:
    mapped_site = map_site_name(site)
    if mapped_site is None:
        print(f"⚠️ Site '{site}' not in site_name_mapping → skipped")
        continue

    site_cont = tb1_cont[tb1_cont[SITE_COL] == site]
    site_cat = tb1_cat[tb1_cat[SITE_COL] == site]
    if site_cont.empty:
        print(f"⚠️ Site '{site}' has zero patients → skipped")
        continue

    mapped_site = mapped_site.replace(' ', '_')
    site_table = build_site_tb1(site_cont, site_cat)
    n_patients = site_table.loc['Total patients', 'Overall']

    out_path = os.path.join(tb1_dir, f"tb1_summary_{mapped_site}.csv")
    site_table.to_csv(out_path)
    print(f"✅ Saved {mapped_site} (n={n_patients}) → {out_path}")
    n_written += 1

print(f"🎉 Done: {n_written} site tables written to {tb1_dir}")